# Presentation visualisations

A separate notebook for model comparisons, colonisation curves and development timelines. No demonstration results are used. The project reference folder was empty when this notebook was created: connect the real exports before interpreting any figures.

**Design:** consistent colours across figures, elapsed days on the growth and timeline axes, colonisation in percent, and self-contained interactive HTML exports. All colours live in the THEME cell and are provisional until the slide colours are supplied.

## Input files
Place CSV files in `presentation_data/` at the project root. Missing files are skipped. Required columns:

| File | Columns |
|---|---|
| `model_metrics.csv` | dataset, split, model, metric, value, run |
| `observations.csv` | dataset, experiment, observation_id, genus, species, oyster_group, temperature_c, days, colonisation_pct |
| `baseline_curves.csv` | dataset, genus, species, oyster_group, temperature_c, days, predicted_pct, model |
| `heldout_predictions.csv` | dataset, split, model, observation_id, days, observed_pct, predicted_pct |
| `app_timeline.csv` | species, common_name, temperature_c, item, kind (stage/event), start_day, end_day, in_range, source |

Use numeric days and temperatures (°C), and percentages in 0–100. `oyster_group` must be `cool`, `tropical`, or `not_oyster`, assigned from experimental metadata. Do not infer oyster categories from names. Baseline curves contain actual model predictions on a day grid, not interpolated observations. Baseline curves are built for every Pleurotus species in the 2.2 bundle at each slider temperature.

Metrics must share evaluation protocols and units within each panel. Supply only out-of-sample predictions for diagnostics; repeated runs should be aggregated first or given additional grouping columns. The app timeline is an estimate from the consensus baseline, not an observation; say so in the final slide captions.

In [ ]:
# THEME: every colour and font used by every figure lives in this cell. Calm earthy palette, provisional until slide colours exist.
THEME = dict(
    background="#F6EFE0",      # cream: plot and paper background
    text="#3B2A1E",            # dark brown: text, axes, outlines
    grid="#E3D6BC",            # light tan
    font="Georgia, 'Times New Roman', serif",
    # bars, lines and series: mushroom browns and moss green
    series=["#7A5C3E", "#5E7336", "#A58562", "#3F5A2A", "#8C6D4B", "#B7A27D"],
    stages={"Colonisation": "#9C7B58",
            "Induction planning marker (2 days)": "#C9B58F",
            "Fruiting (start induction to first harvest)": "#5E7336"},
    # dots and events: dark rust and amber, strong against the browns and the cream
    events={"Inoculation": "#6B2A14", "Fully colonised": "#B5451B", "Start induction": "#D9722B",
            "Pinning": "#E0A030", "First harvest": "#9A5B00"},
    datasets={"d009": "#B5451B", "d025": "#E0A030", "d026": "#6B2A14", "syldd": "#D9722B"},
    dataset_symbols={"d009": "circle", "d025": "diamond", "d026": "square", "syldd": "triangle-up"},
    outline="#FFF8E7",         # thin light edge around dots so they stay visible on bars
    faded_opacity=0.3,
)

In [ ]:
from pathlib import Path
import importlib.util
import re
missing = [m for m in ["pandas", "numpy", "plotly"] if importlib.util.find_spec(m) is None]
if missing:
    raise ImportError("Install notebook dependencies: " + ", ".join(missing))
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "presentation_data"
OUT = ROOT / "presentation_figures"
OUT.mkdir(exist_ok=True)
TEXT, GRID, BACKGROUND = THEME["text"], THEME["grid"], THEME["background"]
SHOW_FIGURES = True
EXPORT_PNG = False  # Requires Kaleido and its browser dependency.

def colours(names):
    names = sorted(set(map(str, names)))
    return {n: THEME["series"][i % len(THEME["series"])] for i, n in enumerate(names)}

def finish(fig, title, filename, height=580, note=None):
    fig.update_layout(title=dict(text=title, x=0.01), template="plotly_white", height=height,
        paper_bgcolor=BACKGROUND, plot_bgcolor=BACKGROUND,
        font=dict(family=THEME["font"], size=16, color=TEXT),
        margin=dict(l=85, r=40, t=100, b=110 if note else 100), legend_title_text="")
    fig.update_xaxes(gridcolor=GRID, linecolor=TEXT, zerolinecolor=GRID)
    fig.update_yaxes(gridcolor=GRID, linecolor=TEXT, zerolinecolor=GRID)
    if note:
        fig.add_annotation(text=note, xref="paper", yref="paper", x=0, y=-0.2, showarrow=False,
                           xanchor="left", font=dict(size=14))
    stem = re.sub(r"[^a-zA-Z0-9_-]+", "_", filename)
    fig.write_html(OUT / (stem + ".html"), include_plotlyjs=True)
    if EXPORT_PNG:
        fig.write_image(OUT / (stem + ".png"), width=1600, height=900, scale=2)
    if SHOW_FIGURES:
        fig.show()
    return fig

In [ ]:
from mycodemo.presentation_data import build_presentation_data

counts, skipped_curves = build_presentation_data(ROOT)
print(counts)
for item in skipped_curves:
    print("Curve skipped:", item)

In [ ]:
SCHEMAS = {
    "model_metrics": "dataset split model metric value run",
    "observations": "dataset experiment observation_id genus species oyster_group temperature_c days colonisation_pct",
    "baseline_curves": "dataset genus species oyster_group temperature_c days predicted_pct model",
    "heldout_predictions": "dataset split model observation_id days observed_pct predicted_pct",
    "app_timeline": "species common_name temperature_c item kind start_day end_day in_range source",
}
NUMERIC = {"value", "temperature_c", "days", "colonisation_pct", "observed_pct", "predicted_pct", "start_day", "end_day", "day"}
UNIQUE_KEYS = {
    "model_metrics": ["dataset", "split", "model", "metric", "run"],
    "observations": ["dataset", "observation_id"],
    "baseline_curves": ["dataset", "genus", "species", "oyster_group", "temperature_c", "days", "model"],
    "heldout_predictions": ["dataset", "split", "model", "observation_id"],
    "app_timeline": ["species", "temperature_c", "item", "start_day"],
}

def load(name):
    columns = SCHEMAS[name].split()
    path = DATA / f"{name}.csv"
    if not path.exists():
        print(f"Awaiting {path}")
        return pd.DataFrame(columns=columns)
    df = pd.read_csv(path)
    absent = set(columns) - set(df.columns)
    if absent:
        raise ValueError(f"{name}: missing columns {sorted(absent)}")
    if df[columns].isna().any().any():
        raise ValueError(f"{name}: required values are missing")
    for col in set(columns) & NUMERIC:
        df[col] = pd.to_numeric(df[col], errors="raise")
        if not np.isfinite(df[col]).all():
            raise ValueError(f"{name}: {col} is not finite")
        if col.endswith("_pct") and not df[col].between(0, 100).all():
            raise ValueError(f"{name}: {col} is outside 0–100; investigate rather than clipping")
        if col in {"days", "start_day", "end_day", "day"} and (df[col] < 0).any():
            raise ValueError(f"{name}: negative elapsed days")
    if "oyster_group" in df and not df.oyster_group.isin(["cool", "tropical", "not_oyster"]).all():
        raise ValueError(f"{name}: invalid oyster_group")
    if name == "app_timeline" and (df.end_day < df.start_day).any():
        raise ValueError("Item ends before it starts")
    if name in UNIQUE_KEYS and df.duplicated(UNIQUE_KEYS[name]).any():
        raise ValueError(f"{name}: duplicate rows for {UNIQUE_KEYS[name]}")
    return df

tables = {name: load(name) for name in SCHEMAS}
model_names = set()
for df in tables.values():
    if "model" in df:
        model_names.update(df.model.astype(str))
model_palette = colours(model_names)
print({name: len(df) for name, df in tables.items()})

## 1. Tested model performance
One panel per model, so datasets, splits and metrics never mix. Every run or fold stays visible as its own point. Coincident values can overlap.

In [ ]:
m = tables["model_metrics"]
panels = [
    ("literature consensus", "Colonisation time estimate: error in days (book values)",
     "Lower is better: absolute error in days when the model reconstructs the book values (not real-world validation).", "strip"),
    ("colonisation forecast test", "Colonisation forecast vs baseline: error by number of photos",
     "Lower is better: forecast error against the number of photos used.", "photos"),
    ("property model", "Material strength: error per held-out lab",
     "Lower is better: error on each lab held out of training.", "strip"),
    ("harvest readiness", "Harvest readiness: mAP50, runs A and B",
     "Higher is better: mAP50 on the validation photos (B = box, M = mask).", "harvest"),
]
for prefix, title, note, kind in panels:
    g = m[m.dataset.str.startswith(prefix)]
    if g.empty:
        print("No data for:", title)
        continue
    if kind == "photos":
        g = g.assign(photos=g.run.astype(float)).sort_values("photos")
        fig = px.line(g, x="photos", y="value", color="model", markers=True, color_discrete_map=model_palette)
        fig.update_xaxes(title="Number of photos")
        fig.update_yaxes(title="Forecast error")
    elif kind == "harvest":
        g = g.assign(kind=g.metric.str.extract(r"\((.)\)")[0].map({"B": "box (B)", "M": "mask (M)"}))
        cmap = dict(zip(sorted(g.kind.unique()), THEME["series"][:2]))
        fig = px.strip(g, x="value", y="split", color="kind", color_discrete_map=cmap, hover_data=["run"])
        fig.update_traces(jitter=0, marker=dict(size=14))
        fig.update_xaxes(title="mAP50", range=[0, 1])
        fig.update_yaxes(title="")
    else:
        fig = px.strip(g, x="value", y="model", color="model", hover_data=["run"], color_discrete_map=model_palette)
        fig.update_traces(jitter=0, marker=dict(size=13))
        fig.update_xaxes(title="Error" if "days" not in title else "Error (days)")
        fig.update_yaxes(title="")
    finish(fig, title, "metrics_" + prefix, note=note)

## 2. Experimental data on the 2.2 baseline
One figure per genus; Pleurotus is split into cool and tropical oysters (`oyster_group`). All datasets share one figure, each with its own marker colour and symbol; hover shows the dataset and substrate. The slider moves in 1 °C steps: it shows the 2.2 baseline curve of every species in the group at that temperature, plus the experimental points whose temperature rounds to that step (syldd 19.85 becomes 20). Tropical species have curves but no experimental dots. Trametes has dots only (no baseline for it).

Baseline curves: logistic rule on the 2.2 estimate (90% at t90, midpoint at 0.60 × t90, a demo assumption), on each species' main substrate group, air temperature. Species and temperatures without a 2.2 number are skipped and listed when the data is built.

In [ ]:
obs, curves = tables["observations"].copy(), tables["baseline_curves"]
obs["step"] = np.floor(obs.temperature_c + 0.5)          # 19.85 -> 20
obs["base"] = obs.dataset.str.split(" · ").str[0]
group_title = {"cool": "cool oysters", "tropical": "tropical oysters", "not_oyster": ""}
contexts = pd.concat([obs[["genus", "oyster_group"]], curves[["genus", "oyster_group"]]]).drop_duplicates()
for genus, og in contexts.itertuples(index=False, name=None):
    o = obs[(obs.genus == genus) & (obs.oyster_group == og)]
    c = curves[(curves.genus == genus) & (curves.oyster_group == og)]
    steps = sorted(set(o.step) | set(c.temperature_c))
    species = sorted(set(o.species) | set(c.species))
    line_colour = colours(species)
    dashes = {s: ["solid", "dash", "dot", "dashdot"][i % 4] for i, s in enumerate(species)}
    fig, trace_steps = go.Figure(), []
    for step in steps:
        for sp, g in c[c.temperature_c == step].groupby("species"):
            g = g.sort_values("days")
            fig.add_trace(go.Scatter(x=g.days, y=g.predicted_pct, mode="lines", name=f"{sp} · 2.2 baseline",
                line=dict(color=line_colour[sp], width=3, dash=dashes[sp]),
                hovertemplate="Day %{x:g}<br>%{y:.1f}%<extra>%{fullData.name}</extra>"))
            trace_steps.append(step)
        for base, g in o[o.step == step].groupby("base"):
            fig.add_trace(go.Scatter(x=g.days, y=g.colonisation_pct, mode="markers", name=base, legendgroup=base,
                marker=dict(color=THEME["datasets"].get(base, THEME["events"]["Fully colonised"]),
                            symbol=THEME["dataset_symbols"].get(base, "circle"), size=10,
                            line=dict(color=THEME["outline"], width=1)),
                customdata=g[["dataset", "species", "observation_id", "temperature_c"]].to_numpy(),
                hovertemplate="%{customdata[0]}<br>%{customdata[1]}<br>Day %{x:g}, %{y:.1f}%<br>%{customdata[3]:.2f} °C"
                              "<br>%{customdata[2]}<extra></extra>"))
            trace_steps.append(step)
    for trace, step in zip(fig.data, trace_steps):
        trace.visible = step == steps[0]
    title = f"{genus}" + (f": {group_title[og]}" if group_title[og] else "") + " · experiments and 2.2 baseline"
    if len(steps) > 1:
        fig.update_layout(sliders=[dict(active=0, currentvalue=dict(prefix="Temperature °C: "),
            steps=[dict(label=f"{s:g}", method="update", args=[{"visible": [t == s for t in trace_steps]}]) for s in steps])])
    xmax = max(float(pd.concat([o.days, c.days]).max()), 1)
    fig.update_xaxes(title="Days", range=[0, xmax * 1.03])
    fig.update_yaxes(title="Colonisation (%)", range=[0, 100])
    if o.empty:
        fig.add_annotation(text="No experimental data for this group", xref="paper", yref="paper", x=0.5, y=0.1,
                           showarrow=False, font=dict(size=14))
    finish(fig, title, f"colonisation_{genus}_{og}", height=640)

## 3. The app's timeline for all species (cultivation + fruiting baseline)
What the app shows a user, produced by calling the app's own code (`pipeline.colonisation_estimate`, `colonisation_forecast` with no photos, `dates_step`, `fruiting_estimate`) for a Food batch on **sawdust** (the app's default; species without a sawdust row use the same fallback as `baseline_guide.row`, noted in the hover). Days are since inoculation. The slider sets the incubation temperature in 1 °C steps.

Bars: colonisation (to the 90% day), the 2-day induction planning marker, fruiting (start induction to first harvest). Dots: the app's dates, drawn at the middle of the app's date range; the whiskers and the hover show the full range. "Fully colonised" is the app's "90% visible (consensus day)" and "Start induction" is its "Induction planning marker". Rows are faded and marked "outside range" when the temperature is outside the species' incubation range. No experimental data here. Species the app has no entry for (P. pulmonarius, G. lucidum) are not shown.

In [ ]:
tl = tables["app_timeline"].copy()
tl["label"] = tl.common_name + " (" + tl.species + ")"
order = list(dict.fromkeys(tl.sort_values(["species"]).label))
temps = sorted(tl.temperature_c.unique())
fig, trace_temps = go.Figure(), []
bar_items = list(THEME["stages"])
event_items = list(THEME["events"])
for temp in temps:
    t = tl[tl.temperature_c == temp]
    for item in bar_items:
        g = t[(t.kind == "stage") & (t["item"] == item)]
        if g.empty:
            continue
        fade = np.where(g.in_range, 1.0, THEME["faded_opacity"])
        hover = np.where(g.in_range, "", "<br><b>outside range</b> (species incubation range)")
        fig.add_trace(go.Bar(y=g.label, x=g.end_day - g.start_day, base=g.start_day, orientation="h", name=item,
            legendgroup=item, width=0.5, marker=dict(color=THEME["stages"][item], opacity=fade),
            customdata=np.c_[g.start_day, g.end_day, hover, g.source],
            hovertemplate="%{y}<br>day %{customdata[0]:.1f} to %{customdata[1]:.1f}%{customdata[2]}"
                          "<br>%{customdata[3]}<extra>%{fullData.name}</extra>"))
        trace_temps.append(temp)
    for item in event_items:
        g = t[(t.kind == "event") & (t["item"] == item)]
        if g.empty:
            continue
        mid = (g.start_day + g.end_day) / 2
        fade = np.where(g.in_range, 1.0, THEME["faded_opacity"])
        hover = np.where(g.in_range, "", "<br><b>outside range</b> (species incubation range)")
        fig.add_trace(go.Scatter(x=mid, y=g.label, mode="markers", name=item, legendgroup=item,
            marker=dict(color=THEME["events"][item], size=13, symbol="diamond", opacity=fade,
                        line=dict(color=THEME["outline"], width=1)),
            error_x=dict(type="data", symmetric=True, array=(g.end_day - g.start_day) / 2, thickness=1.5, width=4,
                         color=THEME["events"][item]),
            customdata=np.c_[g.start_day, g.end_day, hover, g.source],
            hovertemplate="%{y}<br>day %{customdata[0]:.0f} to %{customdata[1]:.0f}%{customdata[2]}"
                          "<br>%{customdata[3]}<extra>%{fullData.name}</extra>"))
        trace_temps.append(temp)
start = min(temps, key=lambda x: abs(x - 22))
for trace, temp in zip(fig.data, trace_temps):
    trace.visible = temp == start
fig.update_layout(barmode="overlay", sliders=[dict(active=temps.index(start), currentvalue=dict(prefix="Temperature °C: "),
    steps=[dict(label=f"{x:g}", method="update", args=[{"visible": [t == x for t in trace_temps]}]) for x in temps])])
fig.update_xaxes(title="Days since inoculation", range=[0, float(tl.end_day.max()) * 1.04])
fig.update_yaxes(title="", categoryorder="array", categoryarray=order, autorange="reversed")
finish(fig, "App timeline by species (consensus baseline, 2.2 colonisation estimate)", "app_timeline",
       height=max(520, 190 + 70 * len(order)))

## 4. Suggested addition: prediction accuracy and errors over time
Observed versus predicted values show calibration against the dashed identity line. Residuals show overprediction (positive) or underprediction (negative), and systematic errors across days. Use held-out experiments where possible to avoid leakage from repeated observations.


In [ ]:
pred = tables["heldout_predictions"].copy()
if not pred.empty:
    pred["residual_pct"] = pred.predicted_pct - pred.observed_pct
    for (dataset, split), g in pred.groupby(["dataset", "split"]):
        fig = px.scatter(g, x="observed_pct", y="predicted_pct", color="model",
            hover_data=["observation_id", "days"], color_discrete_map=model_palette)
        fig.add_shape(type="line", x0=0, x1=100, y0=0, y1=100, line=dict(color=TEXT, dash="dash"))
        fig.update_xaxes(title="Observed colonisation (%)", range=[0, 100])
        fig.update_yaxes(title="Predicted colonisation (%)", range=[0, 100], scaleanchor="x", scaleratio=1)
        finish(fig, f"Held-out predictions · {dataset} · {split}", f"predicted_observed_{dataset}_{split}")
        fig = px.scatter(g, x="days", y="residual_pct", color="model",
            hover_data=["observation_id"], color_discrete_map=model_palette)
        fig.add_hline(y=0, line_dash="dash", line_color=TEXT)
        fig.update_xaxes(title="Days")
        fig.update_yaxes(title="Prediction − observation (percentage points)")
        finish(fig, f"Residuals · {dataset} · {split}", f"residuals_{dataset}_{split}")


## Presentation checklist and further improvements
- Replace the provisional THEME with the slide colours. No scientific results have been verified yet.
- Add sample sizes, data sources, metric units/direction and evaluation split definitions to slide captions.
- Add a temperature-by-species coverage heatmap if sampling is uneven; it helps explain gaps.
- Add replicate-level confidence bands when the experimental design supports them. Do not count repeated daily measurements as independent replicates.
- Identify observed, estimated and censored stage dates. Unknown dates must not become zero.
- For static slides, choose one temperature and use Plotly's camera button. Interactive HTML retains the slider. Optional PNG export requires Kaleido and its browser dependency.

Figures export to `presentation_figures/`. This notebook is an input-ready starting point; wiring it to existing training notebooks and matching the actual deck requires the missing project files.